# A2 — Logit Cross-Section

**Famiglia A — Cross-section** | Una riga per contatore

## Modello
```
silente_prevalente_i = 1  se  pct_silente > 0.5
P(silente_prevalente=1) = σ(β₀ + β·X_i)
```
- Variabile dipendente binaria derivata
- Interpretazione tramite odds ratio

> **TODO**: da implementare

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 1 — Setup
# ══════════════════════════════════════════════════════════════════════════════

from google.colab import drive
drive.mount("/content/drive")

import subprocess, os, sys, importlib

REPO_URL  = "https://github.com/swaggincoffee-bit/Tesi.git"
REPO_PATH = "/content/Tesi"
if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)
sys.path.insert(0, REPO_PATH)

PROC = "/content/drive/MyDrive/Contatori/OUTPUT/"
OUT  = "/content/drive/MyDrive/Contatori/OUTPUT/"

for f in ["df_cs.parquet"]:
    print(f"  {'✅' if os.path.exists(PROC + f) else '❌'} {f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 2 — Import
# ══════════════════════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import LabelEncoder
from src import utils
importlib.reload(utils)

print("✅ Librerie importate correttamente.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 3 — Caricamento dati
# ══════════════════════════════════════════════════════════════════════════════

df_cs = pd.read_parquet(PROC + "df_cs.parquet")
print(f"df_cs: {df_cs.shape}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 4 — Feature engineering
# ══════════════════════════════════════════════════════════════════════════════

df_lett_re = pd.read_parquet(PROC + "df_lett_re.parquet")
ref_date = pd.to_datetime(df_lett_re["data_lettura"]).max()
del df_lett_re

for short, col in {"costruttore_enc":  "costruttore",
                   "modello_enc":      "modello",
                   "tecnologia_enc":   "tecnologia",
                   "comune_enc":       "comune",
                   "accessibile_enc":  "accessibile",
                   "telegestione_enc": "telegestione"}.items():
    df_cs[short] = LabelEncoder().fit_transform(df_cs[col].fillna("N/A"))

FEATURES = [
    "anni_da_costruzione",
    "consumo_annuo",
    "gg_da_installazione",
    "firmware_num",
    "gg_da_ult_com",
    "gg_da_ult_mis",
    "costruttore_enc",
    "modello_enc",
    "tecnologia_enc",
    "comune_enc",
    "accessibile_enc",
    "telegestione_enc",
    "pct_err",
    *[f"diag_bit_{i:02d}" for i in range(16)],
]

df_cs["silente_sempre"] = (df_cs["pct_silente"] == 1.0).astype(int)
TARGET = "silente_sempre"

df_model = df_cs[FEATURES + [TARGET]].dropna()
print(f"Osservazioni: {len(df_model):,}  (NaN scartati: {len(df_cs)-len(df_model):,})")
print(f"\nDistribuzione target:")
print(df_model[TARGET].value_counts())

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 5 — Logit
# ══════════════════════════════════════════════════════════════════════════════

X = sm.add_constant(df_model[FEATURES])
y = df_model[TARGET]

model = sm.Logit(y, X).fit(maxiter=200, method="bfgs", disp=True)
print(model.summary())

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELLA 6 — Odds ratio
# ══════════════════════════════════════════════════════════════════════════════

import numpy as np

risultati = model.summary2().tables[1].reset_index()
risultati.columns = ["variabile", "coef", "std_err", "z", "p_value", "ci_low", "ci_high"]
risultati["odds_ratio"] = np.exp(risultati["coef"])
risultati["or_ci_low"]  = np.exp(risultati["ci_low"])
risultati["or_ci_high"] = np.exp(risultati["ci_high"])

risultati.to_csv(OUT + "A2_coefficienti_Logit.csv", index=False)
print("✅ Salvato A2_coefficienti_Logit.csv")
display(risultati)